# Sesión 8: Ejercicios de visualización

Usa como referencia el notebook `08_visualizacion.ipynb` y el cheat sheet de las diapositivas.

Todas las preguntas usan los datos de la ENDI que ya están cargados en `personas`, `df` y `df_pond`.

## Configuración

In [ ]:
import polars as pl
import pyreadr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess

AZUL    = "#005B85"
NARANJA = "#E8520A"
VERDE   = "#2D9E6B"
DARK    = "#1A1A2E"
MID     = "#4A5568"
LIGHT   = "#F7F9FC"
PALETA_CAT = [AZUL, NARANJA, VERDE, "#9B59B6", "#E67E22", "#1ABC9C"]

plt.rcParams.update({
    "axes.facecolor":    LIGHT,
    "figure.facecolor":  LIGHT,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#DDE3EA",
    "grid.linewidth":    0.6,
    "axes.labelsize":    11,
    "axes.titlesize":    12,
})
sns.set_theme(style="whitegrid", palette=PALETA_CAT, font_scale=1.0)

ruta = r"..\..\Materiales\Insumos\ENDI\BDD_ENDI_R2_rds\BDD_ENDI_R2_f1_personas.rds"
personas = pl.from_pandas(pyreadr.read_r(ruta)[None])
personas = (
    personas
    .select(["id_upm", "estrato", "fexp", "area", "region", "etnia",
             "f1_s1_2", "edaddias",
             "f1_s5_5_1", "f1_s5_5_2", "f1_s5_5_3",
             "f1_s5_6_1", "f1_s5_6_2", "f1_s5_6_3",
             "dcronica", "quintil", "pobreza"])
    .rename({
        "f1_s1_2":   "sexo",
        "f1_s5_5_1": "long1", "f1_s5_5_2": "long2", "f1_s5_5_3": "long3",
        "f1_s5_6_1": "tal1",  "f1_s5_6_2": "tal2",  "f1_s5_6_3": "tal3",
    })
    .filter((pl.col("edaddias") < 1826) & pl.col("dcronica").is_not_null())
    .with_columns(
        pl.when(pl.col("edaddias") < 730)
        .then(pl.mean_horizontal("long1", "long2", "long3"))
        .otherwise(pl.mean_horizontal("tal1", "tal2", "tal3"))
        .alias("talla")
    )
    .drop(["long1", "long2", "long3", "tal1", "tal2", "tal3"])
    .with_columns([
        pl.when(pl.col("area")    == 1).then(pl.lit("urbano"))
          .otherwise(pl.lit("rural")).alias("area"),
        pl.when(pl.col("sexo")    == 1).then(pl.lit("hombre"))
          .otherwise(pl.lit("mujer")).alias("sexo"),
        pl.when(pl.col("region")  == 1).then(pl.lit("sierra"))
          .when(pl.col("region")  == 2).then(pl.lit("costa"))
          .when(pl.col("region")  == 3).then(pl.lit("amazonia"))
          .otherwise(pl.lit(None)).alias("region"),
        pl.when(pl.col("pobreza") == 1).then(pl.lit("no pobre"))
          .otherwise(pl.lit("pobre")).alias("pobreza"),
    ])
)

df = personas.to_pandas()

def ponderar(df, col_peso="fexp"):
    """Devuelve un DataFrame con filas replicadas según fexp/fexp.min()."""
    pesos = df[col_peso].to_numpy()
    repeticiones = np.round(pesos / pesos.min()).astype(int)
    indices = np.repeat(np.arange(len(df)), repeticiones)
    return df.iloc[indices].reset_index(drop=True)

df_pond = ponderar(df)
print("Listo.")

---
## Ejercicio 1: Una variable continua

Grafica la distribución de `edaddias` con histograma y KDE, ponderado por `fexp`.

**Pregunta:** ¿la distribución es uniforme entre 0 y 1825 días o hay concentraciones en algún rango de edad?

In [ ]:
# Crea una figura con dos paneles: sin ponderar a la izquierda, ponderado a la derecha
# Usa sns.histplot con bins=50 y kde=True
# En el panel ponderado pasa weights="fexp"
# Etiqueta los ejes y agrega un titulo a cada panel
# Responde la pregunta en un comentario al final de la celda


---
## Ejercicio 2: Variable continua × variable categórica

Compara la distribución de `talla` entre las tres regiones (`sierra`, `costa`, `amazonia`) usando violin por grupo, ponderado.

**Pregunta:** ¿en qué región es menor la talla mediana?

In [ ]:
# Usa sns.violinplot con x="region", y="talla", hue="region"
# Pasa data=df_pond para incorporar la ponderación
# Usa inner="box" para ver la mediana dentro del violin
# Controla el orden de las regiones con el parámetro order
# Responde la pregunta en un comentario al final de la celda


---
## Ejercicio 3: Una variable categórica

Calcula y grafica la prevalencia ponderada de `pobreza` por `region` con barras verticales.

**Pregunta:** ¿en qué región es mayor la prevalencia de pobreza? Ordena las barras de mayor a menor.

In [ ]:
# Calcula primero en Polars: filtra nulos, agrupa por region
# La prevalencia ponderada es (sum(indicador * fexp)) / sum(fexp)
# Donde indicador es 1 si pobreza == "pobre" y 0 si no
# Ordena el resultado de mayor a menor antes de graficar
# Usa ax.bar y formatea el eje y con mticker.PercentFormatter(xmax=1)
# Responde la pregunta en un comentario al final de la celda


---
## Ejercicio 4: Dos variables continuas

Grafica `talla` vs `edaddias` con lowess ponderado y regresión lineal en el mismo panel.

**Pregunta:** ¿en qué rango de edad la regresión lineal sobreestima o subestima más la tendencia real?

In [ ]:
# Prepara sub con personas.select(["edaddias", "talla"]).drop_nulls().to_pandas()
# Prepara sub_pond con df_pond[["edaddias", "talla"]].dropna()
# Calcula la regresion lineal con np.polyfit(x, y, deg=1) y evalua con np.polyval
# Calcula el lowess sin ponderar con lowess(y, x, frac=0.2, return_sorted=True)
# Calcula el lowess ponderado con los mismos parametros pero usando sub_pond
# Grafica scatter de puntos (alpha bajo), linea lineal y curva lowess en el mismo ax
# Agrega leyenda que distinga los tres elementos
# Responde la pregunta en un comentario al final de la celda


---
## Ejercicio 5: Multivariado

Reproduce el scatter de `talla` vs `edaddias` incorporando tres variables visuales adicionales:
- **Color hue:** `area` (urbano / rural)
- **Size:** `quintil` (1 al 5)
- **Alpha:** `dcronica` (opaco si tiene desnutrición, transparente si no)

**Pregunta:** ¿qué patrón observas en los puntos con `dcronica = 1` respecto a la talla esperada para su edad?

In [ ]:
# Filtra df_pond para quedarte con las columnas necesarias y elimina nulos
# Toma una muestra de 6000 filas con sample(n=6000, random_state=42)
# Define color_map para area: {"urbano": AZUL, "rural": NARANJA}
# Define size_map para quintil: {1: 20, 2: 40, 3: 65, 4: 95, 5: 130}
# Dibuja primero los puntos con dcronica==0 con alpha=0.10
# Dibuja encima los puntos con dcronica==1 con alpha=0.75
# Agrega leyendas manuales con Patch para color y Line2D para size
# Responde la pregunta en un comentario al final de la celda
